In [2]:
"""
Single-Player Analysis Pipeline
WhoScored / Opta Event Data  ·  Full Solo Analysis

Outputs (10 visuals):
  01_touch_map.png                 – KDE heatmap of all touches
  02_runs_final_third.png          – off-ball runs into final third or beyond
  03_chance_creation.png           – key passes + goal assists (open play)
  04_takeon_map.png                – take-on map: success vs failure
  05_post_takeon_action.png        – post-dribble pass arrows coloured by xT gained
  06_progressive_carries.png       – progressive carries with speed overlay
  07_xt_<player>.png               – xT map (passes + crosses) net xT per zone
  08_pass_connections_ft.png       – passing connections ending in final third or beyond
  09_foot_split.png                – right vs left foot %
  10_crosses_into_box.png          – crosses into the penalty area
  11_summary_radar.png             – single-page summary radar / stat block

Coordinate system: Opta – x: 0→100 (own→opp goal), y: 0→100 (bottom→top).
"""

# !pip install mplsoccer matplotlib pandas numpy selenium webdriver-manager scipy -q

import json
import time
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from mplsoccer import Pitch

warnings.filterwarnings('ignore')


# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  —  edit only this section
# ════════════════════════════════════════════════════════════════════════════

BG_COLOR    = '#FFF8E7'
TEXT_COLOR  = '#1A1A1A'
PITCH_LINE  = '#C4A882'
GOLD        = '#F6A623'
GREY        = '#AAAAAA'
CROSS_COLOR = '#00BCD4'          # accent colour for crosses throughout
CREDIT_LINE = 'Data: Opta | xTransfers Episode 36'

COMPETITION = '2025/26 Season'

PLAYER = {
    'name':           'Yan Diomande',
    'player_id':      557000,
    'team':           'RB Leipzig',
    'color':          '#0A2463',
    'match_prefix':   ['1908', '1910'],
    'match_suffixes': [
        [319],
        [607, 620, 639, 661, 626, 681, 689, 720, 738, 758, 767, 793, 654, 
         802, 810, 819, 684, 829, 834, 711, 727, 745, 764, 787, 837, 848, 
         854, 861, 873, 876, 891, 898],
    ],
}

# Build flat match-ID list
_ids = []
for _prefix, _suffixes in zip(PLAYER['match_prefix'], PLAYER['match_suffixes']):
    if _prefix == '':
        _ids.extend(int(s) for s in _suffixes)
    else:
        _ids.extend(int(f"{_prefix}{str(s).zfill(3)}") for s in _suffixes)
PLAYER['match_ids'] = _ids

OUTPUT_DIR = Path(f'./analysis_{PLAYER["name"].lower().replace(" ", "_")}')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Player : {PLAYER['name']}  ({PLAYER['team']})")
print(f"ID     : {PLAYER['player_id']}")
print(f"Matches: {len(PLAYER['match_ids'])}")


# ════════════════════════════════════════════════════════════════════════════
# OPTA EVENT HELPERS
# ════════════════════════════════════════════════════════════════════════════

SHOT_TYPES        = {10, 11, 12, 13, 14, 16}
POSSESSION_EVENTS = {1, 3, 10, 11, 12, 13, 14, 16}
QUALIFIER_OVERRUN    = 211
QUALIFIER_CROSS      = 2
QUALIFIER_HEAD       = 15
QUALIFIER_RIGHT_FOOT = 20
QUALIFIER_LEFT_FOOT  = 72
QUALIFIER_FREEKICK   = 5       # FreekickTaken   — on pass events
QUALIFIER_CORNER     = 6       # CornerTaken     — on pass events
QUALIFIER_THROWIN    = 107     # ThrowIn         — on pass events
QUALIFIER_GOALKICK   = 124     # GoalKick        — on pass events

# Qualifier 210 sits on the PASS event: 'The pass was an assist for a shot.'
QUALIFIER_PASS_ASSIST = 210

# Set-piece qualifier IDs — used to define open play
SET_PIECE_QS = frozenset({
    QUALIFIER_FREEKICK,    # 5
    QUALIFIER_CORNER,      # 6
    QUALIFIER_THROWIN,     # 107
    QUALIFIER_GOALKICK,    # 124
})

OPT_TO_M_X = 105.0 / 100.0
OPT_TO_M_Y = 68.0  / 100.0


def get_type_value(ev):
    t = ev.get('type', {})
    return t.get('value', -1) if isinstance(t, dict) else (int(t) if t else -1)

def is_successful(ev):
    oc = ev.get('outcomeType', {})
    return (oc.get('value', 0) if isinstance(oc, dict) else 0) == 1

def has_qualifier(ev, qid):
    return any(q.get('type', {}).get('value') == qid for q in ev.get('qualifiers', []))

def is_open_play(ev):
    """True if the event did NOT originate from a dead-ball set piece."""
    return not any(has_qualifier(ev, qid) for qid in SET_PIECE_QS)

def event_time(ev):
    return ev.get('minute', 0) * 60.0 + ev.get('second', 0.0)

def xy(ev):        return ev.get('x'),    ev.get('y')
def end_xy(ev):    return ev.get('endX'), ev.get('endY')
def _safe_xs(evs): return [e['x'] for e in evs if e.get('x') is not None]
def _safe_ys(evs): return [e['y'] for e in evs if e.get('y') is not None]

def get_foot(ev):
    if has_qualifier(ev, QUALIFIER_RIGHT_FOOT): return 'right'
    if has_qualifier(ev, QUALIFIER_LEFT_FOOT):  return 'left'
    return None

def carry_distance_m(c):
    dx = (c['endX'] - c['x']) * OPT_TO_M_X
    dy = (c['endY'] - c['y']) * OPT_TO_M_Y
    return (dx**2 + dy**2) ** 0.5

def carry_progression_m(c):
    return (c['endX'] - c['x']) * OPT_TO_M_X

def carry_speed_mps(c):
    dt = c.get('dt_s', 0.1)
    return carry_distance_m(c) / dt if dt > 0 else 0.0

def chain_leads_to_shot(sorted_events, start_idx, team_id, max_team_actions=3, max_time=15.0):
    if team_id is None:
        return False
    t0 = event_time(sorted_events[start_idx])
    actions_counted = 0
    for ev in sorted_events[start_idx + 1:]:
        if event_time(ev) - t0 > max_time: break
        ev_team = ev.get('teamId')
        if ev_team is not None and ev_team != team_id: break
        if ev_team != team_id: continue
        tv = get_type_value(ev)
        if tv not in POSSESSION_EVENTS: continue
        if tv in SHOT_TYPES: return True
        actions_counted += 1
        if actions_counted >= max_team_actions: break
    return False

print('Event helpers ready.')


# ════════════════════════════════════════════════════════════════════════════
# xT — KARUN SINGH 16×12 GRID
# ════════════════════════════════════════════════════════════════════════════

XT_GRID = np.array([
    [0.000,0.000,0.001,0.001,0.001,0.002,0.002,0.003,0.004,0.006,0.008,0.013,0.022,0.042,0.068,0.112],
    [0.000,0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.008,0.012,0.019,0.032,0.062,0.094,0.143],
    [0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.007,0.011,0.016,0.024,0.040,0.072,0.113,0.168],
    [0.000,0.001,0.001,0.002,0.003,0.003,0.004,0.006,0.008,0.013,0.019,0.028,0.046,0.082,0.128,0.184],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.005,0.007,0.009,0.014,0.021,0.031,0.052,0.090,0.138,0.198],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.006,0.008,0.011,0.016,0.023,0.035,0.057,0.098,0.153,0.224],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.006,0.008,0.011,0.016,0.023,0.035,0.057,0.098,0.153,0.224],
    [0.000,0.001,0.001,0.002,0.003,0.004,0.005,0.007,0.009,0.014,0.021,0.031,0.052,0.090,0.138,0.198],
    [0.000,0.001,0.001,0.002,0.003,0.003,0.004,0.006,0.008,0.013,0.019,0.028,0.046,0.082,0.128,0.184],
    [0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.007,0.011,0.016,0.024,0.040,0.072,0.113,0.168],
    [0.000,0.000,0.001,0.001,0.002,0.002,0.003,0.004,0.005,0.008,0.012,0.019,0.032,0.062,0.094,0.143],
    [0.000,0.000,0.001,0.001,0.001,0.002,0.002,0.003,0.004,0.006,0.008,0.013,0.022,0.042,0.068,0.112],
])

N_ROWS, N_COLS = XT_GRID.shape
CELL_W = 100 / N_COLS
CELL_H = 100 / N_ROWS

def get_xt_zone(x, y):
    col = min(int(x / 100 * N_COLS), N_COLS - 1)
    row = min(int(y / 100 * N_ROWS), N_ROWS - 1)
    return col, row

def get_xt_value(x, y):
    col, row = get_xt_zone(x, y)
    return float(XT_GRID[row, col])

def compute_pass_cross_xt(succ_passes):
    results = []
    for ev in succ_passes:
        sx, sy = xy(ev); ex, ey = end_xy(ev)
        if None in (sx, sy, ex, ey): continue
        xt_s = get_xt_value(sx, sy); xt_e = get_xt_value(ex, ey)
        results.append({'x': sx, 'y': sy, 'endX': ex, 'endY': ey,
                        'xt_start': xt_s, 'xt_end': xt_e,
                        'xt_gained': xt_e - xt_s,
                        'is_cross': has_qualifier(ev, QUALIFIER_CROSS)})
    return results

def compute_xt_zones_gross(all_xt):
    zone = np.zeros((N_ROWS, N_COLS))
    for d in all_xt:
        col, row = get_xt_zone(d['x'], d['y'])
        zone[row, col] += d['xt_end']
    return zone

def compute_xt_zones_net(all_xt):
    zone = np.zeros((N_ROWS, N_COLS))
    for d in all_xt:
        col, row = get_xt_zone(d['x'], d['y'])
        zone[row, col] += d['xt_gained']
    return zone

def compute_xt_zones_count(all_xt):
    zone = np.zeros((N_ROWS, N_COLS), dtype=int)
    for d in all_xt:
        col, row = get_xt_zone(d['x'], d['y'])
        zone[row, col] += 1
    return zone

print('xT helpers ready.')


# ════════════════════════════════════════════════════════════════════════════
# FILTER + INFERENCE FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def filter_passes(player_events):
    passes = [ev for ev in player_events if get_type_value(ev) == 1]
    return ([ev for ev in passes if is_successful(ev)],
            [ev for ev in passes if not is_successful(ev)])

def split_passes_crosses(succ_passes):
    crosses = [ev for ev in succ_passes if has_qualifier(ev, QUALIFIER_CROSS)]
    passes  = [ev for ev in succ_passes if not has_qualifier(ev, QUALIFIER_CROSS)]
    return passes, crosses

def filter_takeons(player_events):
    tos = [ev for ev in player_events
           if get_type_value(ev) == 3 and not has_qualifier(ev, QUALIFIER_OVERRUN)]
    return ([ev for ev in tos if is_successful(ev)],
            [ev for ev in tos if not is_successful(ev)])

def reconstruct_carries(all_events, player_id):
    player_evs = [ev for ev in all_events
                  if ev.get('playerId') == player_id
                  and get_type_value(ev) in POSSESSION_EVENTS]
    player_evs.sort(key=event_time)
    carries = []
    for i in range(len(player_evs) - 1):
        curr, nxt = player_evs[i], player_evs[i + 1]
        ex, ey = end_xy(curr); nx, ny = xy(nxt)
        if None in (ex, ey, nx, ny): continue
        dt = event_time(nxt) - event_time(curr)
        if dt < 0 or dt > 12: continue
        dist = ((ex - nx)**2 + (ey - ny)**2) ** 0.5
        if dist < 2.5 or dist > 35.0: continue
        carries.append({'x': ex, 'y': ey, 'endX': nx, 'endY': ny,
                        'distance': dist, 'dt_s': max(dt, 0.1)})
    return carries

def is_progressive_carry(c):              return (c['endX'] - c['x']) >= 10.0
def is_carry_into_box(c):                 return c['x'] < 83.0 and c['endX'] >= 83.0 and 21.1 <= c['endY'] <= 78.9
def is_carry_to_final_third_or_beyond(c): return c['endX'] >= 66.67

def detect_off_ball_runs(all_events, player_id):
    player_evs = [ev for ev in all_events
                  if ev.get('playerId') == player_id
                  and get_type_value(ev) in POSSESSION_EVENTS]
    player_evs.sort(key=event_time)
    runs = []
    for i in range(1, len(player_evs)):
        prev, curr = player_evs[i - 1], player_evs[i]
        rx, ry = xy(curr)
        if rx is None or rx <= 66.67: continue
        px, py = xy(prev)
        if px is None or px >= 66.67: continue
        dt = event_time(curr) - event_time(prev)
        if dt < 2 or dt > 25: continue
        runs.append({'from_x': px, 'from_y': py, 'to_x': rx, 'to_y': ry, 'dt_s': dt})
    return runs

def filter_chance_creations_open(player_events, all_events):
    """
    Open-play chance creation events (key passes + goal assists).
    Uses qualifier 210 on pass events — the correct Opta pass-level marker.
    Returns a list of enriched dicts.
    """
    sorted_all = sorted(all_events, key=event_time)
    ev_by_id = {}
    for ev in sorted_all:
        eid = ev.get('id') or ev.get('eventId')
        if eid:
            ev_by_id[eid] = ev

    results = []
    for ev in player_events:
        if get_type_value(ev) != 1:                        continue
        if not is_open_play(ev):                           continue
        if not has_qualifier(ev, QUALIFIER_PASS_ASSIST):   continue

        sx, sy = xy(ev);  ex, ey = end_xy(ev)
        if None in (sx, sy, ex, ey):                       continue

        is_goal_assist = False

        related_id = ev.get('relatedEventId')
        if related_id and related_id in ev_by_id:
            if get_type_value(ev_by_id[related_id]) == 16:
                is_goal_assist = True
        else:
            t0 = event_time(ev)
            for nxt in sorted_all:
                nt = event_time(nxt)
                if nt <= t0:                    continue
                if nt - t0 > 5.0:              break
                if nxt.get('playerId') == ev.get('playerId'): continue
                tv = get_type_value(nxt)
                if tv in SHOT_TYPES:
                    is_goal_assist = (tv == 16)
                    break

        results.append({
            'ev':             ev,
            'sx': sx, 'sy': sy, 'ex': ex, 'ey': ey,
            'is_goal_assist': is_goal_assist,
            'is_key_pass':    not is_goal_assist,
            'into_box':       ex >= 83.0 and 21.1 <= ey <= 78.9,
            'xt_gained':      get_xt_value(ex, ey) - get_xt_value(sx, sy),
        })
    return results


def post_takeon_actions(all_events, player_id, window=5):
    sorted_all = sorted(all_events, key=event_time)
    results = []
    for i, ev in enumerate(sorted_all):
        if ev.get('playerId') != player_id: continue
        if get_type_value(ev) != 3 or not is_successful(ev): continue
        if has_qualifier(ev, QUALIFIER_OVERRUN): continue
        to_x, to_y = end_xy(ev)
        if to_x is None: to_x, to_y = xy(ev)
        if to_x is None: continue
        t_to = event_time(ev)
        for j in range(i + 1, len(sorted_all)):
            nxt = sorted_all[j]
            if event_time(nxt) - t_to > window: break
            if nxt.get('playerId') != player_id: continue
            t_nxt = get_type_value(nxt)
            if t_nxt == 1:
                ex, ey = end_xy(nxt)
                if None not in (ex, ey):
                    xt_s = get_xt_value(to_x, to_y)
                    xt_e = get_xt_value(ex, ey)
                    results.append({'to_x': to_x, 'to_y': to_y,
                                    'action': 'pass', 'endX': ex, 'endY': ey,
                                    'success': is_successful(nxt),
                                    'leads_to_shot': chain_leads_to_shot(
                                        sorted_all, j, nxt.get('teamId'),
                                        max_team_actions=2, max_time=15.0),
                                    'xt_start': xt_s, 'xt_end': xt_e,
                                    'xt_gained': xt_e - xt_s})
                break
            elif t_nxt in SHOT_TYPES:
                results.append({'to_x': to_x, 'to_y': to_y, 'action': 'shot',
                                 'endX': to_x, 'endY': to_y, 'success': t_nxt == 16})
                break
            elif t_nxt in POSSESSION_EVENTS:
                nx, ny = xy(nxt)
                if None not in (nx, ny):
                    if ((to_x - nx)**2 + (to_y - ny)**2) ** 0.5 >= 2.5:
                        results.append({'to_x': to_x, 'to_y': to_y, 'action': 'carry',
                                         'endX': nx, 'endY': ny, 'success': True})
                break
    return results


def passing_connections_final_third(all_events, player_id, player_name_map):
    sorted_all = sorted(all_events, key=event_time)
    team_id = None
    for ev in sorted_all:
        if ev.get('playerId') == player_id and ev.get('teamId'):
            team_id = ev['teamId']
            break

    results = []
    for i, ev in enumerate(sorted_all):
        if ev.get('playerId') != player_id: continue
        if get_type_value(ev) != 1: continue
        if not is_successful(ev): continue

        sx, sy = xy(ev); ex, ey = end_xy(ev)
        if None in (sx, sy, ex, ey): continue
        if ex < 66.67: continue

        is_cross  = has_qualifier(ev, QUALIFIER_CROSS)
        xt_gained = get_xt_value(ex, ey) - get_xt_value(sx, sy)

        recipient_id = ev.get('relatedPlayerId')
        if not recipient_id and team_id:
            t0 = event_time(ev)
            for nxt in sorted_all[i + 1:]:
                if event_time(nxt) - t0 > 5.0: break
                ev_team = nxt.get('teamId')
                if ev_team is None: continue
                if ev_team != team_id: break
                r_id = nxt.get('playerId')
                if r_id and r_id != player_id and get_type_value(nxt) in POSSESSION_EVENTS:
                    recipient_id = r_id
                    break

        if not recipient_id:
            continue

        recipient_name = player_name_map.get(int(recipient_id),
                                             player_name_map.get(recipient_id,
                                             f'#{recipient_id}'))
        results.append({
            'sx': sx, 'sy': sy, 'ex': ex, 'ey': ey,
            'recipient_id':   recipient_id,
            'recipient_name': recipient_name,
            'xt_gained':      xt_gained,
            'xt_gross':       get_xt_value(ex, ey),   # destination xT value (gross)
            'is_cross':       is_cross,
        })

    return results


def compute_foot_split(acc):
    actions   = acc['succ_passes'] + acc['fail_passes'] + acc['shots']
    right     = sum(1 for ev in actions if get_foot(ev) == 'right')
    left      = sum(1 for ev in actions if get_foot(ev) == 'left')
    untagged  = len(actions) - right - left
    total_tag = right + left
    return {'right': right, 'left': left, 'untagged': untagged,
            'total_tagged': total_tag,
            'pct_right': right / total_tag * 100 if total_tag else 0.0,
            'pct_left':  left  / total_tag * 100 if total_tag else 0.0}

print('Filter functions ready.')


# ════════════════════════════════════════════════════════════════════════════
# SELENIUM DRIVER + WHOSCORED SCRAPER
# ════════════════════════════════════════════════════════════════════════════

def get_driver():
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager
    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver


def scrape_match(driver, match_id, player_id, player_name):
    url = f'https://www.whoscored.com/Matches/{match_id}/Live'
    print(f'  [WS] Fetching match {match_id} ...')
    driver.get(url)
    time.sleep(random.uniform(10, 16))
    src = driver.page_source

    marker_pos = src.find('matchCentreData')
    if marker_pos == -1:
        print(f'       matchCentreData not found — skipping {match_id}')
        return [], None, {}

    json_start = src.find('{', marker_pos)
    if json_start == -1:
        print(f'       No opening brace found in match {match_id}')
        return [], None, {}

    depth = 0; in_string = False; escape = False; end_idx = json_start
    for i, ch in enumerate(src[json_start:], json_start):
        if escape:                     escape = False; continue
        if ch == '\\' and in_string:  escape = True;  continue
        if ch == '"' and not escape:  in_string = not in_string; continue
        if in_string:                 continue
        if ch == '{':                 depth += 1
        elif ch == '}':               depth -= 1
        if depth == 0:                end_idx = i; break

    try:
        data = json.loads(src[json_start: end_idx + 1])
    except Exception as e:
        print(f'       Parse error {match_id}: {e}')
        return [], None, {}

    events = data.get('events', [])
    if not events:
        return [], None, {}

    player_name_map = {}
    found_name, player_team_id = None, None
    for side in ('home', 'away'):
        side_data = data.get(side, {})
        for pl in side_data.get('players', []):
            pid = pl.get('playerId') or pl.get('id')
            pname = pl.get('name', f'#{pid}')
            if pid:
                player_name_map[int(pid)] = pname
            if pid == player_id:
                found_name     = pname
                player_team_id = side_data.get('teamId')

    if not found_name:
        print(f'       {player_name} not in squad — skipping {match_id}')
        return [], None, {}

    for ev in events:
        ev['match_id'] = match_id

    n = sum(1 for ev in events if ev.get('playerId') == player_id)
    print(f'       ✓ {found_name} — {n} raw events')
    return events, player_team_id, player_name_map


def process_match(all_events, player_id, player_name_map):
    player_evs = [ev for ev in all_events if ev.get('playerId') == player_id]

    succ_passes, fail_passes = filter_passes(player_evs)
    to_succ, to_fail         = filter_takeons(player_evs)
    carries                  = reconstruct_carries(all_events, player_id)
    prog_carries             = [c for c in carries if is_progressive_carry(c)]
    box_carries              = [c for c in carries if is_carry_into_box(c)]
    shots                    = [ev for ev in player_evs if get_type_value(ev) in SHOT_TYPES]
    runs                     = detect_off_ball_runs(all_events, player_id)
    post_to                  = post_takeon_actions(all_events, player_id)
    pass_conns               = passing_connections_final_third(all_events, player_id, player_name_map)
    chances                  = filter_chance_creations_open(player_evs, all_events)

    return {
        'succ_passes': succ_passes, 'fail_passes': fail_passes,
        'to_succ': to_succ, 'to_fail': to_fail,
        'carries': carries, 'prog_carries': prog_carries, 'box_carries': box_carries,
        'shots': shots, 'runs': runs,
        'post_takeon': post_to,
        'pass_connections': pass_conns,
        'chances': chances,
    }


def scrape_player(driver, player_cfg):
    acc = {
        'succ_passes': [], 'fail_passes': [],
        'to_succ': [], 'to_fail': [],
        'carries': [], 'prog_carries': [], 'box_carries': [],
        'shots': [], 'runs': [],
        'post_takeon': [],
        'pass_connections': [],
        'chances': [],
    }
    games = 0
    for mid in player_cfg['match_ids']:
        all_evs, _, name_map = scrape_match(
            driver, mid, player_cfg['player_id'], player_cfg['name'])
        if not all_evs:
            continue
        m = process_match(all_evs, player_cfg['player_id'], name_map)
        for k in acc:
            acc[k].extend(m.get(k, []))
        games += 1
        time.sleep(random.uniform(10, 18))
    print(f"  → {player_cfg['name']}: {games} games successfully scraped")
    return acc, games

print('Scraper ready.')


# ════════════════════════════════════════════════════════════════════════════
# PITCH + LAYOUT HELPERS
# ════════════════════════════════════════════════════════════════════════════

def _pitch():
    return Pitch(pitch_type='opta', pitch_color=BG_COLOR,
                 line_color=PITCH_LINE, linewidth=1.4,
                 goal_type='box', corner_arcs=True)

def _make_fig(title, subtitle, note='', figsize=(22, 14)):
    """Single-panel figure with full-width pitch."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    fig.set_facecolor(BG_COLOR)
    fig.subplots_adjust(top=0.88, bottom=0.02, left=0.02, right=0.92)
    p = _pitch()
    p.draw(ax=ax)
    ax.set_facecolor(BG_COLOR)

    y0 = 1.00
    fig.text(0.5, y0,       title,
             ha='center', va='bottom', fontsize=32, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.5, y0 - 0.045, subtitle,
             ha='center', va='bottom', fontsize=20, color=TEXT_COLOR, alpha=0.90)
    fig.text(0.5, y0 - 0.082, CREDIT_LINE,
             ha='center', va='bottom', fontsize=13, color=TEXT_COLOR, alpha=0.85, style='italic')
    if note:
        fig.text(0.5, y0 - 0.115, note,
                 ha='center', va='bottom', fontsize=12, color=TEXT_COLOR, alpha=0.85, style='italic')
    return fig, ax


def _stats_box(ax, txt, color=None):
    if color is None: color = TEXT_COLOR
    ax.text(0.02, 0.97, txt, transform=ax.transAxes, ha='left', va='top',
            fontsize=16, color=TEXT_COLOR,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=BG_COLOR,
                      edgecolor=color, alpha=0.95),
            fontfamily='monospace')

print('Layout helpers ready.')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 1 — TOUCH MAP
# ════════════════════════════════════════════════════════════════════════════

def _collect_touches(acc):
    all_evs = (acc['succ_passes'] + acc['fail_passes']
               + acc['to_succ'] + acc['to_fail'] + acc['shots'])
    pts = [(e['x'], e['y']) for e in all_evs if e.get('x') is not None]
    pts += [(c['x'], c['y']) for c in acc['carries'] if c.get('x') is not None]
    return pts

def plot_touch_map(acc, n):
    color = PLAYER['color']
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Touch Map",
        subtitle = f"{COMPETITION}  ·  {PLAYER['team']}  ·  KDE heatmap of all actionable touches  ·  attacking left → right",
        note     = 'Passes · carries · shots · take-ons',
    )
    pts = _collect_touches(acc)
    if not pts:
        ax.text(50, 50, 'No data', ha='center', va='center', fontsize=20, color=TEXT_COLOR)
    else:
        xs, ys = zip(*pts)
        p = _pitch()
        p.kdeplot(xs, ys, ax=ax, fill=True, levels=100,
                  cmap='YlOrRd', alpha=0.84, zorder=2, bw_adjust=0.85)
        p.scatter(xs, ys, ax=ax, s=4, color=color, alpha=0.08, zorder=3)
        _stats_box(ax, f'{len(pts)} touches  ·  {n} games', color=color)
    fig.savefig(OUTPUT_DIR / '01_touch_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 01_touch_map.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 2 — OFF-BALL RUNS INTO FINAL THIRD / BEYOND
# ════════════════════════════════════════════════════════════════════════════

def plot_runs_final_third(acc, n):
    color = PLAYER['color']
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Runs into Final Third / Beyond",
        subtitle = f"{COMPETITION}  ·  {PLAYER['team']}  ·  attacking left → right",
        note     = 'Arrow = inferred run (prior touch → reception in final third or beyond)  ·  Gold ● = reception  ·  Gold ★ = in box  ·  Gap: 2–25 s',
    )
    runs = acc['runs']
    ax.add_patch(plt.Rectangle((66.67, 0), 33.33, 100, color='#4A90D9', alpha=0.05, zorder=1))
    ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8, color=color, alpha=0.07, zorder=1))
    in_box = [r for r in runs if r['to_x'] >= 83.0 and 21.1 <= r['to_y'] <= 78.9]
    other  = [r for r in runs if r not in in_box]
    for r in other:
        p.arrows(r['from_x'], r['from_y'], r['to_x'], r['to_y'], ax=ax,
                 color=color, width=1.6, headwidth=4.5, headlength=4.5, alpha=0.52, zorder=3)
    for r in in_box:
        p.arrows(r['from_x'], r['from_y'], r['to_x'], r['to_y'], ax=ax,
                 color=color, width=2.2, headwidth=5.5, headlength=5.5, alpha=0.85, zorder=4)
    if other:
        p.scatter([r['to_x'] for r in other], [r['to_y'] for r in other], ax=ax,
                  s=70, color=GOLD, edgecolors=color, linewidths=0.7, alpha=0.82, zorder=5)
    if in_box:
        p.scatter([r['to_x'] for r in in_box], [r['to_y'] for r in in_box], ax=ax,
                  s=220, color=GOLD, marker='*', edgecolors=color,
                  linewidths=0.8, alpha=0.97, zorder=6)
    legend = [Line2D([0],[0], marker='>', color=color, lw=0, markersize=12,
                     label=f'Runs into final third: {len(other)}'),
              Line2D([0],[0], marker='*', color=GOLD,  lw=0, markersize=16,
                     label=f'Runs into box: {len(in_box)}')]
    ax.legend(handles=legend, loc='lower left', fontsize=15,
              facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
    _stats_box(ax, f'{len(runs)} total runs  ·  {len(in_box)} into box  ·  {n} games', color=color)
    fig.savefig(OUTPUT_DIR / '02_runs_final_third.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 02_runs_final_third.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 3 — CHANCE CREATION MAP  (open play only)
#
# Key Pass  (qualifier 210, no goal chain): player colour arrow
# Goal Assist (qualifier 210 + goal chain): GOLD star at destination
# ════════════════════════════════════════════════════════════════════════════

GA_COLOR = GOLD           # goal assist arrow

def plot_chance_creation(acc, n):
    color   = PLAYER['color']
    chances = acc['chances']
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Chance Creation Map",
        subtitle = f"{COMPETITION}  ·  {PLAYER['team']}  ·  Open-play key passes + goal assists  ·  attacking left → right",
        note     = (
            'Player colour = key pass (shot did not result in goal)  ·  '
            'Gold ★ = goal assist  ·  '
            'Qualifier 210 (PassAssist) on pass events  ·  '
            'Open play only'
        ),
    )

    ax.add_patch(plt.Rectangle((66.67, 0), 33.33, 100,
                 color='#4A90D9', alpha=0.04, zorder=1, linewidth=0))
    ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                 color=color, alpha=0.06, zorder=1, linewidth=0))
    ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                 fill=False, edgecolor=color, linewidth=1.1,
                 linestyle='--', alpha=0.30, zorder=2))

    if not chances:
        ax.text(50, 50, 'No chance-creation events found', ha='center',
                va='center', fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'0 chances created  ·  {n} games', color=color)
    else:
        key_passes   = [c for c in chances if     c['is_key_pass']]
        goal_assists = [c for c in chances if     c['is_goal_assist']]
        into_box_all = [c for c in chances if     c['into_box']]

        # ── Key passes (player colour) ─────────────────────────────────────
        for c in key_passes:
            p.arrows(c['sx'], c['sy'], c['ex'], c['ey'], ax=ax,
                     color=TEXT_COLOR, width=3.2, headwidth=7.0,
                     headlength=7.0, alpha=0.35, zorder=3)
            p.arrows(c['sx'], c['sy'], c['ex'], c['ey'], ax=ax,
                     color=color, width=2.0, headwidth=4.8,
                     headlength=4.8, alpha=0.90, zorder=4)
            p.scatter(c['ex'], c['ey'], ax=ax, s=90, color=color,
                      edgecolors=BG_COLOR, linewidths=0.8,
                      alpha=0.85, zorder=5)

        # ── Goal assists (gold, star at destination) ───────────────────────
        for c in goal_assists:
            p.arrows(c['sx'], c['sy'], c['ex'], c['ey'], ax=ax,
                     color=TEXT_COLOR, width=4.0, headwidth=9.0,
                     headlength=9.0, alpha=0.45, zorder=5)
            p.arrows(c['sx'], c['sy'], c['ex'], c['ey'], ax=ax,
                     color=GA_COLOR, width=2.6, headwidth=6.0,
                     headlength=6.0, alpha=0.97, zorder=6)
            p.scatter(c['ex'], c['ey'], ax=ax, s=440, color=GA_COLOR,
                      marker='*', edgecolors=TEXT_COLOR, linewidths=1.0,
                      alpha=0.97, zorder=7)

        # ── Origin dots ────────────────────────────────────────────────────
        p.scatter([c['sx'] for c in chances],
                  [c['sy'] for c in chances], ax=ax, s=60,
                  color=TEXT_COLOR, edgecolors=BG_COLOR,
                  linewidths=0.7, alpha=0.40, zorder=4)

        total  = len(chances)
        per_g  = total / n if n else 0.0
        xt_tot = sum(c['xt_gained'] for c in chances)
        xt_avg = xt_tot / total if total else 0.0

        legend = [
            Line2D([0],[0], marker='>', color=color, lw=0, markersize=13,
                   label=f'Key pass: {len(key_passes)}'),
            Line2D([0],[0], marker='*', color=GA_COLOR, lw=0, markersize=18,
                   markeredgecolor=TEXT_COLOR, markeredgewidth=0.8,
                   label=f'Goal assist: {len(goal_assists)}'),
            Line2D([0],[0], marker='o', color=TEXT_COLOR, lw=0,
                   markersize=9, alpha=0.40, label='Origin of chance'),
        ]
        ax.legend(handles=legend, loc='lower left', fontsize=14,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
            f'Chances created (open play) : {total}\n'
            f'  Key passes                : {len(key_passes)}\n'
            f'  Goal assists              : {len(goal_assists)}\n'
            f'  Ending in box             : {len(into_box_all)}\n'
            f'Per game                    : {per_g:.2f}\n'
            f'────────────────────────────\n'
            f'Total xT from chances       : {xt_tot:+.3f}\n'
            f'Avg xT per chance           : {xt_avg:+.4f}\n'
            f'Games                       : {n}',
            color=color)

    fig.savefig(OUTPUT_DIR / '03_chance_creation.png',
                dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 03_chance_creation.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — TAKE-ON MAP
# ════════════════════════════════════════════════════════════════════════════

def plot_takeon_map(acc, n):
    color = PLAYER['color']
    FAIL_COLOR = '#FF6D00'
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Take-On Map",
        subtitle = f"{COMPETITION}  ·  {PLAYER['team']}  ·  attacking left → right",
        note     = 'Player colour ● = successful dribble  ·  Contrasting ✕ = failed dribble  ·  KDE = dribble density',
    )
    succ = acc['to_succ']; fail = acc['to_fail']; all_tos = succ + fail
    if not all_tos:
        ax.text(50, 50, 'No data', ha='center', va='center', fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'0 take-ons  ·  {n} games', color=color)
    else:
        all_xs = _safe_xs(all_tos); all_ys = _safe_ys(all_tos)
        if len(all_xs) >= 4:
            p.kdeplot(all_xs, all_ys, ax=ax, fill=True, levels=50,
                      cmap='YlOrRd', alpha=0.22, zorder=2, bw_adjust=1.10)
        if fail:
            p.scatter(_safe_xs(fail), _safe_ys(fail), ax=ax, s=230, color=FAIL_COLOR,
                      marker='X', edgecolors='white', linewidths=0.7, alpha=0.92, zorder=4)
        if succ:
            p.scatter(_safe_xs(succ), _safe_ys(succ), ax=ax, s=250, color=color,
                      marker='o', edgecolors='white', linewidths=0.8, alpha=0.92, zorder=5)
        total = len(all_tos); sr = len(succ) / total * 100 if total else 0
        legend = [Line2D([0],[0], marker='o', color=color, lw=0, markersize=15,
                         markeredgecolor='white', markeredgewidth=0.8,
                         label=f'Successful: {len(succ)}'),
                  Line2D([0],[0], marker='X', color=FAIL_COLOR, lw=0, markersize=12,
                         markeredgecolor='white', markeredgewidth=0.7,
                         label=f'Failed: {len(fail)}')]
        ax.legend(handles=legend, loc='lower left', fontsize=15,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax, f'{total} take-ons  ·  {sr:.0f}% success  ·  {n} games', color=color)
    fig.savefig(OUTPUT_DIR / '04_takeon_map.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 04_takeon_map.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — POST-TAKE-ON ACTION MAP
# ════════════════════════════════════════════════════════════════════════════

_XT_PASS_CMAP = LinearSegmentedColormap.from_list(
    'xt_pass', ['#C62828', '#F9A825', '#2E7D32'], N=256)
CHAIN_TO_SHOT_COLOR = '#E91E63'
CARRY_ACCENT_COLOR  = '#9C27B0'

def plot_post_takeon_action(acc, n):
    color = PLAYER['color']
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Post-Take-On Action (≤ 5 s)",
        subtitle = f"{COMPETITION}  ·  {PLAYER['team']}  ·  attacking left → right",
        note     = ('Arrow = pass coloured by xT gained (green + / red −)  ·  '
                    'SCA outline = pass → shot ≤ 2 team actions  ·  '
                    'Dashed = carry  ·  Gold ★ = shot  ·  Origin = take-on endpoint'),
    )
    actions = acc['post_takeon']
    passes  = [a for a in actions if a['action'] == 'pass']
    carries = [a for a in actions if a['action'] == 'carry']
    shots   = [a for a in actions if a['action'] == 'shot']
    chain_passes = [a for a in passes if a.get('leads_to_shot')]

    if not actions:
        ax.text(50, 50, 'No data', ha='center', va='center', fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'0 post-take-on actions  ·  {n} games', color=color)
    else:
        xt_vals = [a.get('xt_gained', 0.0) for a in passes]
        abs_max = max((abs(v) for v in xt_vals), default=0.001)
        norm_xt = Normalize(vmin=-abs_max, vmax=abs_max)

        if passes + carries:
            p.scatter([a['to_x'] for a in passes + carries],
                      [a['to_y'] for a in passes + carries],
                      ax=ax, s=90, color=TEXT_COLOR, edgecolors=BG_COLOR,
                      linewidths=1.2, alpha=0.50, zorder=2)

        for a in passes:
            xt    = a.get('xt_gained', 0.0)
            rgba  = _XT_PASS_CMAP(norm_xt(xt))
            ratio = abs(xt) / (abs_max + 1e-9)
            w_core = 1.4 + 2.0 * ratio
            p.arrows(a['to_x'], a['to_y'], a['endX'], a['endY'], ax=ax,
                     color=TEXT_COLOR, width=w_core + 1.6,
                     headwidth=(w_core + 1.6) * 2.6, headlength=(w_core + 1.6) * 2.6,
                     alpha=0.35, zorder=3)
            p.arrows(a['to_x'], a['to_y'], a['endX'], a['endY'], ax=ax,
                     color=rgba, width=w_core, headwidth=w_core * 2.8,
                     headlength=w_core * 2.8, alpha=0.96, zorder=4)
            if a.get('leads_to_shot'):
                p.arrows(a['to_x'], a['to_y'], a['endX'], a['endY'], ax=ax,
                         color=CHAIN_TO_SHOT_COLOR, width=0.5,
                         headwidth=w_core * 3.8, headlength=0,
                         alpha=0.88, zorder=4.5)

        for a in carries:
            for clr, lw, al in [(TEXT_COLOR, 3.6, 0.35), (CARRY_ACCENT_COLOR, 2.4, 0.95)]:
                ax.annotate('', xy=(a['endX'], a['endY']), xytext=(a['to_x'], a['to_y']),
                            arrowprops=dict(arrowstyle='-|>', color=clr, lw=lw,
                                            linestyle='dashed', alpha=al,
                                            connectionstyle='arc3,rad=0'), zorder=5)

        if shots:
            p.scatter([a['to_x'] for a in shots], [a['to_y'] for a in shots],
                      ax=ax, s=380, color=GOLD, marker='*',
                      edgecolors=TEXT_COLOR, linewidths=1.5, alpha=0.97, zorder=7)

        sm = ScalarMappable(cmap=_XT_PASS_CMAP, norm=norm_xt)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.018, pad=0.02, shrink=0.50)
        cbar.set_label('xT gained per pass', fontsize=12, color=TEXT_COLOR, labelpad=6)
        cbar.ax.tick_params(labelcolor=TEXT_COLOR, labelsize=10)
        cbar.outline.set_edgecolor(PITCH_LINE)

        total_xt   = sum(xt_vals)
        avg_xt     = total_xt / len(xt_vals) if xt_vals else 0.0
        pos_passes = sum(1 for v in xt_vals if v > 0)
        neg_passes = sum(1 for v in xt_vals if v < 0)
        max_xt     = max(xt_vals, default=0.0)

        legend = [mpatches.Patch(facecolor='#2E7D32', label='High +xT pass'),
                  mpatches.Patch(facecolor='#C62828', label='Negative xT pass'),
                  Line2D([0],[0], marker='>', color=CHAIN_TO_SHOT_COLOR, lw=0, markersize=14,
                         label=f'SCA → shot ≤2 actions ({len(chain_passes)})'),
                  Line2D([0],[0], marker='>', color=CARRY_ACCENT_COLOR, lw=0, markersize=13,
                         label=f'Carry ({len(carries)})'),
                  Line2D([0],[0], marker='*', color=GOLD, lw=0, markersize=17,
                         markeredgecolor=TEXT_COLOR, markeredgewidth=1.0,
                         label=f'Shot ({len(shots)})')]
        ax.legend(handles=legend, loc='lower left', fontsize=13,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
            f'Post-TO actions  : {len(actions)}\n'
            f'Pass : {len(passes)}  Carry : {len(carries)}  Shot : {len(shots)}\n'
            f'SCA (→ shot ≤2 actions) : {len(chain_passes)}\n'
            f'────────────────────────────\n'
            f'xT from post-TO passes\n'
            f'  Total xT gained : {total_xt:+.4f}\n'
            f'  Avg  xT / pass  : {avg_xt:+.4f}\n'
            f'  Best single pass: {max_xt:+.4f}\n'
            f'  Fwd (+xT)       : {pos_passes} / {len(passes)}\n'
            f'  Back (−xT)      : {neg_passes} / {len(passes)}\n'
            f'Games : {n}',
            color=color)

    fig.savefig(OUTPUT_DIR / '05_post_takeon_action.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 05_post_takeon_action.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — PROGRESSIVE CARRIES
# ════════════════════════════════════════════════════════════════════════════

def plot_progressive_carries(acc, n):
    color = PLAYER['color']
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Progressive Carries",
        subtitle = f"{COMPETITION}  ·  {PLAYER['team']}  ·  Carry advancing ≥ 10 Opta units forward  ·  attacking left → right",
        note     = 'Gradient = carry speed (m/s)  ·  Gold = carry ending in box',
    )
    prog  = acc['prog_carries']; box_c = acc['box_carries']
    box_set = set(id(c) for c in box_c)
    non_box = [c for c in prog if id(c) not in box_set]
    if not prog:
        ax.text(50, 50, 'No data', ha='center', va='center', fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'0 progressive carries  ·  {n} games', color=color)
    else:
        speeds   = [carry_speed_mps(c) for c in prog]
        max_spd  = max(speeds) if speeds else 1.0
        avg_spd  = sum(speeds) / len(speeds) if speeds else 0.0
        progressions = [carry_progression_m(c) for c in prog]
        avg_prog_m   = sum(progressions) / len(progressions) if progressions else 0.0
        distances    = [carry_distance_m(c) for c in prog]
        longest_m    = max(distances) if distances else 0.0

        speed_cmap = LinearSegmentedColormap.from_list(
            'spd', [mcolors.to_rgba(color, 0.18), mcolors.to_rgba(color, 1.0)], N=256)
        ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                     color=color, alpha=0.06, zorder=1, linewidth=0))
        ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                     fill=False, edgecolor=color, linewidth=1.0,
                     linestyle='--', alpha=0.30, zorder=2))

        non_box_spds = [carry_speed_mps(c) for c in non_box]
        for c, spd in zip(non_box, non_box_spds):
            ratio = spd / (max_spd + 1e-9)
            w = 1.2 + 1.6 * ratio; al = 0.38 + 0.52 * ratio
            p.arrows(c['x'], c['y'], c['endX'], c['endY'], ax=ax,
                     color=speed_cmap(ratio), width=w,
                     headwidth=w * 2.6, headlength=w * 2.6, alpha=al, zorder=3)
        for c in box_c:
            p.arrows(c['x'], c['y'], c['endX'], c['endY'], ax=ax,
                     color=GOLD, width=2.4, headwidth=6.0, headlength=6.0, alpha=0.96, zorder=5)
            p.scatter(c['endX'], c['endY'], ax=ax, s=220, color=GOLD, marker='*',
                      edgecolors=color, linewidths=0.8, alpha=0.97, zorder=6)

        sm = ScalarMappable(cmap=speed_cmap, norm=Normalize(vmin=0, vmax=max_spd))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.020, pad=0.02, shrink=0.55)
        cbar.set_label('Carry speed (m/s)', fontsize=12, color=TEXT_COLOR, labelpad=6)
        cbar.ax.tick_params(labelcolor=TEXT_COLOR, labelsize=11)
        cbar.outline.set_edgecolor(PITCH_LINE)

        legend = [Line2D([0],[0], marker='>', color=color, lw=0, markersize=12,
                         label=f'Progressive: {len(non_box)}'),
                  Line2D([0],[0], marker='*', color=GOLD, lw=0, markersize=14,
                         label=f'Into box: {len(box_c)}')]
        ax.legend(handles=legend, loc='lower left', fontsize=14,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
            f'Progressive carries : {len(prog)}\n'
            f'Into box            : {len(box_c)}\n'
            f'Avg carry speed     : {avg_spd:.2f} m/s\n'
            f'Max carry speed     : {max_spd:.2f} m/s\n'
            f'Avg progression     : {avg_prog_m:.1f} m/carry\n'
            f'Longest carry       : {longest_m:.1f} m\n'
            f'Games               : {n}',
            color=color)

    fig.savefig(OUTPUT_DIR / '06_progressive_carries.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 06_progressive_carries.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — xT CREATED MAP
# ════════════════════════════════════════════════════════════════════════════

def plot_xt_solo(acc, n_games):
    color = PLAYER['color']
    succ_passes = acc['succ_passes']
    if not succ_passes:
        print(f"  [!] No passes — xT plot skipped")
        return

    reg_passes, crosses = split_passes_crosses(succ_passes)
    pass_xt  = compute_pass_cross_xt(reg_passes)
    cross_xt = compute_pass_cross_xt(crosses)
    all_xt   = pass_xt + cross_xt

    zone_gross = compute_xt_zones_gross(all_xt)
    zone_count = compute_xt_zones_count(all_xt)

    total_gross = sum(d['xt_end']    for d in all_xt)
    pass_gross  = sum(d['xt_end']    for d in pass_xt)
    cross_gross = sum(d['xt_end']    for d in cross_xt)

    slug = PLAYER['name'].lower().replace(' ', '_')
    x_grid = np.linspace(0, 100, N_COLS + 1)
    y_grid = np.linspace(0, 100, N_ROWS + 1)
    cx     = (x_grid[:-1] + x_grid[1:]) / 2.0
    cy     = (y_grid[:-1] + y_grid[1:]) / 2.0

    fig, ax = plt.subplots(1, 1, figsize=(22, 13))
    fig.set_facecolor(BG_COLOR)
    fig.subplots_adjust(top=0.88, bottom=0.02, left=0.02, right=0.92)
    p = _pitch(); p.draw(ax=ax); ax.set_facecolor(BG_COLOR)

    fig.text(0.5, 1.00,
             f"{PLAYER['name']}  ({PLAYER['team']})  —  xT Created via Passes & Crosses per Zone",
             ha='center', va='bottom', fontsize=28, fontweight='bold', color=color)
    fig.text(0.5, 0.955,
             f"{COMPETITION}  |  {n_games} games  |  Total Gross xT: {total_gross:.3f}",
             ha='center', va='bottom', fontsize=18, color=TEXT_COLOR, alpha=0.90)
    fig.text(0.5, 0.918,
             'Karun Singh xT model  ·  GROSS = Σ xT(destination) per origin zone  ·  '
             'Zone label: gross xT / pass count  ·  attacking left → right',
             ha='center', va='bottom', fontsize=11, color=TEXT_COLOR, alpha=0.85, style='italic')
    fig.text(0.5, 0.883, CREDIT_LINE,
             ha='center', va='bottom', fontsize=13, color=TEXT_COLOR, alpha=0.85, style='italic')
    ax.set_title('GROSS xT per origin zone + top passes/crosses',
                 fontsize=16, fontweight='bold', color=color, alpha=0.90, pad=10)

    pos_vals  = zone_gross[zone_gross > 0]
    vmax_gross = float(np.percentile(pos_vals, 95)) if len(pos_vals) > 0 else 1.0
    cmap_gross = LinearSegmentedColormap.from_list('xtg', [BG_COLOR, '#FDD835', color], N=256)
    stats      = dict(statistic=zone_gross, x_grid=x_grid, y_grid=y_grid, cx=cx, cy=cy)
    pcm        = p.heatmap(stats, ax=ax, cmap=cmap_gross, vmin=0, vmax=vmax_gross, alpha=0.88, zorder=2)

    cbar = fig.colorbar(pcm, ax=ax, fraction=0.022, pad=0.02, shrink=0.68)
    cbar.set_label('Cumulative Gross xT (per zone)', fontsize=14, color=TEXT_COLOR, labelpad=8)
    cbar.ax.tick_params(labelcolor=TEXT_COLOR, labelsize=13)
    cbar.outline.set_edgecolor(PITCH_LINE)

    for col_i in range(N_COLS + 1):
        ax.axvline(col_i * CELL_W, color=PITCH_LINE, alpha=0.15, lw=0.5, zorder=3)
    for row_i in range(N_ROWS + 1):
        ax.axhline(row_i * CELL_H, color=PITCH_LINE, alpha=0.15, lw=0.5, zorder=3)

    gross_threshold = vmax_gross * 0.05
    for row_i in range(N_ROWS):
        for col_i in range(N_COLS):
            gross_val = zone_gross[row_i, col_i]
            count_val = zone_count[row_i, col_i]
            if count_val == 0: continue
            brightness = gross_val / (vmax_gross + 1e-9)
            txt_c = 'white' if brightness > 0.55 else TEXT_COLOR
            if gross_val >= gross_threshold:
                ax.text(cx[col_i], cy[row_i] + CELL_H * 0.13,
                        f'{gross_val:.3f}',
                        ha='center', va='center', fontsize=7.5, color=txt_c,
                        fontfamily='monospace', fontweight='bold', alpha=0.94, zorder=5)
            ax.text(cx[col_i], cy[row_i] - CELL_H * 0.22,
                    f'n={count_val}',
                    ha='center', va='center', fontsize=6.0, color=txt_c,
                    fontfamily='monospace', alpha=0.80, zorder=5)

    top_passes = sorted([d for d in pass_xt if d['xt_gained'] > 0],
                        key=lambda d: d['xt_gained'], reverse=True)[:20]
    for d in top_passes:
        p.arrows(d['x'], d['y'], d['endX'], d['endY'], ax=ax,
                 color='#1A1A1A', width=3.4, headwidth=7.5, headlength=7.5, alpha=0.65, zorder=6)
        p.arrows(d['x'], d['y'], d['endX'], d['endY'], ax=ax,
                 color=color, width=2.0, headwidth=5.0, headlength=5.0, alpha=0.95, zorder=7)

    top_crosses = sorted([d for d in cross_xt if d['xt_gained'] > 0],
                         key=lambda d: d['xt_gained'], reverse=True)[:15]
    for d in top_crosses:
        p.arrows(d['x'], d['y'], d['endX'], d['endY'], ax=ax,
                 color='#1A1A1A', width=3.4, headwidth=7.5, headlength=7.5, alpha=0.65, zorder=6)
        p.arrows(d['x'], d['y'], d['endX'], d['endY'], ax=ax,
                 color=CROSS_COLOR, width=2.0, headwidth=5.0, headlength=5.0, alpha=0.95, zorder=7)

    min_p = top_passes[-1]['xt_gained']  if top_passes  else 0
    min_c = top_crosses[-1]['xt_gained'] if top_crosses else 0
    legend = [Line2D([0],[0], marker='>', color=color,       lw=0, markersize=13,
                     markeredgecolor='#1A1A1A', markeredgewidth=0.8,
                     label=f'Top-20 passes by xT gained  (min {min_p:.4f})'),
              Line2D([0],[0], marker='>', color=CROSS_COLOR, lw=0, markersize=13,
                     markeredgecolor='#1A1A1A', markeredgewidth=0.8,
                     label=f'Top-15 crosses by xT gained  (min {min_c:.4f})')]
    ax.legend(handles=legend, loc='lower left', fontsize=11,
              facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)

    active = int((zone_gross > 0).sum())
    _stats_box(ax,
        f'Gross xT (passes)  : {pass_gross:.3f}\n'
        f'Gross xT (crosses) : {cross_gross:.3f}\n'
        f'Total Gross xT     : {total_gross:.3f}\n'
        f'────────────────────────\n'
        f'Succ. passes       : {len(reg_passes)}\n'
        f'Crosses            : {len(crosses)}\n'
        f'Active zones       : {active}',
        color=color)

    filename = f'07_xt_{slug}.png'
    fig.savefig(OUTPUT_DIR / filename, dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print(f'  ✓ {filename}  (pass xT: {pass_gross:.3f} | cross xT: {cross_gross:.3f} | total: {total_gross:.3f})')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — PASSING CONNECTIONS IN FINAL THIRD / BEYOND
#
# Stat box shows:
#   • Top 3 by frequency
#   • Top 3 by avg xT
#   • Top 3 by total (gross) xT  ← new
# ════════════════════════════════════════════════════════════════════════════

def _aggregate_connections(pass_conns, min_count=2, top_n=10):
    groups = defaultdict(lambda: {
        'name': '', 'count': 0, 'pass_count': 0, 'cross_count': 0,
        'total_xt': 0.0, 'total_xt_gross': 0.0,
        'sx_list': [], 'sy_list': [], 'ex_list': [], 'ey_list': [],
    })
    for r in pass_conns:
        rid = r['recipient_id']
        g   = groups[rid]
        g['name']           = r['recipient_name']
        g['count']         += 1
        g['total_xt']      += r['xt_gained']
        g['total_xt_gross'] += r.get('xt_gross', 0.0)
        g['sx_list'].append(r['sx']); g['sy_list'].append(r['sy'])
        g['ex_list'].append(r['ex']); g['ey_list'].append(r['ey'])
        if r.get('is_cross'):
            g['cross_count'] += 1
        else:
            g['pass_count']  += 1

    result = []
    for rid, g in groups.items():
        if g['count'] < min_count: continue
        result.append({
            'recipient_id':   rid,
            'recipient_name': g['name'],
            'count':          g['count'],
            'pass_count':     g['pass_count'],
            'cross_count':    g['cross_count'],
            'total_xt':       g['total_xt'],
            'total_xt_gross': g['total_xt_gross'],
            'avg_xt':         g['total_xt'] / g['count'],
            'avg_sx': np.mean(g['sx_list']), 'avg_sy': np.mean(g['sy_list']),
            'avg_ex': np.mean(g['ex_list']), 'avg_ey': np.mean(g['ey_list']),
            'sx_list': g['sx_list'], 'sy_list': g['sy_list'],
            'ex_list': g['ex_list'], 'ey_list': g['ey_list'],
        })

    result.sort(key=lambda x: x['count'], reverse=True)
    return result[:top_n]


def plot_passing_connections_final_third(acc, n):
    color      = PLAYER['color']
    pass_conns = acc['pass_connections']
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Passing & Crossing Connections: Final Third & Beyond",
        subtitle = (f"{COMPETITION}  ·  {PLAYER['team']}  ·  "
                    "Successful passes + crosses ending in final third (x ≥ 66.67)  ·  attacking left → right"),
        note     = ('Solid thin = pass  ·  Dashed thin = cross  ·  Bold arrow = combined summary  ·  '
                    'Arrow width ∝ frequency  ·  Arrow colour = avg xT  ·  ● = avg reception  ·  min 2 to appear'),
    )

    if not pass_conns:
        ax.text(50, 50, 'No final-third data', ha='center', va='center',
                fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'0 final-third actions  ·  {n} games', color=color)
        fig.savefig(OUTPUT_DIR / '08_pass_connections_ft.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
        plt.close(); print('  ✓ 08_pass_connections_ft.png'); return

    conns = _aggregate_connections(pass_conns, min_count=2, top_n=10)
    if not conns:
        ax.text(50, 50, 'No repeated connections', ha='center', va='center',
                fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'{len(pass_conns)} actions  ·  no pair ≥ 2  ·  {n} games', color=color)
        fig.savefig(OUTPUT_DIR / '08_pass_connections_ft.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
        plt.close(); print('  ✓ 08_pass_connections_ft.png'); return

    # Shade zones
    ax.add_patch(plt.Rectangle((66.67, 0), 33.33, 100,
                 color='#4A90D9', alpha=0.06, zorder=1, linewidth=0))
    ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                 color=color, alpha=0.06, zorder=1, linewidth=0))

    # Individual thin lines
    top_ids = {c['recipient_id'] for c in conns}
    for r in pass_conns:
        if r['recipient_id'] not in top_ids: continue
        if r.get('is_cross'):
            ax.plot([r['sx'], r['ex']], [r['sy'], r['ey']],
                    color=CROSS_COLOR, alpha=0.18, linewidth=0.9,
                    linestyle='--', dashes=(3, 3), zorder=2)
        else:
            ax.plot([r['sx'], r['ex']], [r['sy'], r['ey']],
                    color=color, alpha=0.10, linewidth=0.8, zorder=2)

    # Summary arrows
    avg_xt_vals = [c['avg_xt'] for c in conns]
    max_avg_xt  = max(avg_xt_vals) if avg_xt_vals else 0.001
    min_avg_xt  = min(avg_xt_vals) if avg_xt_vals else 0.0
    conn_cmap   = LinearSegmentedColormap.from_list('conn', [BG_COLOR, GOLD, color], N=256)
    norm_conn   = Normalize(vmin=min_avg_xt, vmax=max_avg_xt)
    max_count   = max(c['count'] for c in conns)

    for c in conns:
        rgba = conn_cmap(norm_conn(c['avg_xt']))
        w    = 1.8 + 3.7 * (c['count'] / max_count)
        p.arrows(c['avg_sx'], c['avg_sy'], c['avg_ex'], c['avg_ey'], ax=ax,
                 color=TEXT_COLOR, width=w + 1.4,
                 headwidth=(w + 1.4) * 2.5, headlength=(w + 1.4) * 2.5,
                 alpha=0.35, zorder=3)
        p.arrows(c['avg_sx'], c['avg_sy'], c['avg_ex'], c['avg_ey'], ax=ax,
                 color=rgba, width=w, headwidth=w * 2.8, headlength=w * 2.8,
                 alpha=0.96, zorder=4)
        if c['cross_count'] > c['pass_count']:
            p.arrows(c['avg_sx'], c['avg_sy'], c['avg_ex'], c['avg_ey'], ax=ax,
                     color=CROSS_COLOR, width=0.5, headwidth=w * 3.4, headlength=0,
                     alpha=0.75, zorder=4.5)

    # Reception dots
    for c in conns:
        dot_size = 60 + 220 * (c['count'] / max_count)
        rgba     = conn_cmap(norm_conn(c['avg_xt']))
        p.scatter(c['avg_ex'], c['avg_ey'], ax=ax,
                  s=dot_size, color=rgba,
                  edgecolors=TEXT_COLOR, linewidths=1.2, alpha=0.97, zorder=5)

    # Name badges
    for c in conns:
        lx = min(max(c['avg_ex'], 3), 97)
        ly = min(max(c['avg_ey'] + 3.5, 3), 97)
        p_str = f"{c['pass_count']}p" if c['pass_count'] else ''
        c_str = f"{c['cross_count']}c" if c['cross_count'] else ''
        combo = '  '.join(filter(None, [p_str, c_str]))
        badge_txt = f"{c['recipient_name']}\n{combo}  ·  {c['avg_xt']:+.3f} xT"
        ax.text(lx, ly, badge_txt,
                ha='center', va='bottom', fontsize=8.5, color=TEXT_COLOR,
                fontfamily='monospace', fontweight='bold', zorder=7,
                bbox=dict(boxstyle='round,pad=0.25', facecolor=BG_COLOR,
                          edgecolor=color, alpha=0.88, linewidth=0.8))

    legend = [
        Line2D([0],[0], color=color, lw=1.5, alpha=0.55,   label='Individual pass (solid)'),
        Line2D([0],[0], color=CROSS_COLOR, lw=1.5, alpha=0.70,
               linestyle='--', label='Individual cross (dashed)'),
        mpatches.Patch(facecolor=conn_cmap(1.0), edgecolor=TEXT_COLOR, label='High avg-xT connection'),
        mpatches.Patch(facecolor=conn_cmap(0.0), edgecolor=TEXT_COLOR, label='Low avg-xT connection'),
    ]
    ax.legend(handles=legend, loc='lower left', fontsize=12,
              facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)

    sm = ScalarMappable(cmap=conn_cmap, norm=norm_conn)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.018, pad=0.02, shrink=0.50)
    cbar.set_label('Avg xT per connection', fontsize=12, color=TEXT_COLOR, labelpad=6)
    cbar.ax.tick_params(labelcolor=TEXT_COLOR, labelsize=10)
    cbar.outline.set_edgecolor(PITCH_LINE)

    # ── Stat box: top 3 by count / avg xT / gross xT ──────────────────────
    total_passes  = sum(1 for r in pass_conns if not r.get('is_cross'))
    total_crosses = sum(1 for r in pass_conns if r.get('is_cross'))
    total_ft_xt   = sum(r['xt_gained'] for r in pass_conns)

    top_by_count = conns[:3]                                             # already sorted by count
    top_by_avg   = sorted(conns, key=lambda c: c['avg_xt'],        reverse=True)[:3]
    top_by_gross = sorted(conns, key=lambda c: c['total_xt_gross'], reverse=True)[:3]

    def _fmt_row(c, metric):
        name = c['recipient_name'][:16]
        if metric == 'count':
            return f"  {name:<16} {c['pass_count']}p {c['cross_count']}c  ({c['count']} total)"
        elif metric == 'avg':
            return f"  {name:<16} {c['avg_xt']:+.3f}  ({c['count']}×)"
        else:  # gross
            return f"  {name:<16} {c['total_xt_gross']:.3f}  ({c['count']}×)"

    by_count_str = '\n'.join(_fmt_row(c, 'count') for c in top_by_count)
    by_avg_str   = '\n'.join(_fmt_row(c, 'avg')   for c in top_by_avg)
    by_gross_str = '\n'.join(_fmt_row(c, 'gross') for c in top_by_gross)

    _stats_box(ax,
        f'FT+ passes         : {total_passes}\n'
        f'FT+ crosses        : {total_crosses}\n'
        f'Net xT             : {total_ft_xt:+.3f}\n'
        f'Unique recipients  : {len(set(r["recipient_id"] for r in pass_conns))}\n'
        f'─────────────────────────────────\n'
        f'Top 3 by frequency (p=pass c=cross):\n'
        f'{by_count_str}\n'
        f'─────────────────────────────────\n'
        f'Top 3 by avg xT per pass:\n'
        f'{by_avg_str}\n'
        f'─────────────────────────────────\n'
        f'Top 3 by total gross xT:\n'
        f'{by_gross_str}\n'
        f'Games : {n}',
        color=color)

    fig.savefig(OUTPUT_DIR / '08_pass_connections_ft.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 08_pass_connections_ft.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — RIGHT vs LEFT FOOT SPLIT
# ════════════════════════════════════════════════════════════════════════════

def plot_foot_split(acc, n):
    color = PLAYER['color']
    fs = compute_foot_split(acc)
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))
    fig.set_facecolor(BG_COLOR)
    fig.subplots_adjust(top=0.72, bottom=0.26, left=0.08, right=0.92)
    fig.text(0.5, 0.97, f"{PLAYER['name']}  ·  {PLAYER['team']}  —  Right vs Left Foot Split",
             ha='center', va='top', fontsize=22, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.5, 0.88,
             f"{COMPETITION}  ·  Shots + passes (succ. & failed)  ·  qualifiers 20 (R) / 72 (L)  ·  "
             'untagged actions excluded from %',
             ha='center', va='top', fontsize=13, color=TEXT_COLOR, alpha=0.90)
    fig.text(0.5, 0.80, CREDIT_LINE, ha='center', va='top',
             fontsize=11, color=TEXT_COLOR, alpha=0.85, style='italic')
    ax.set_facecolor(BG_COLOR)
    if fs['total_tagged'] == 0:
        ax.axis('off')
        ax.text(0.5, 0.5, 'No foot-tagged actions', ha='center', va='center',
                fontsize=14, color=TEXT_COLOR, transform=ax.transAxes)
    else:
        ax.barh(0, fs['pct_left'],  color=color, edgecolor=TEXT_COLOR, linewidth=0.8,
                height=0.55, label=f"Left ({fs['left']})")
        ax.barh(0, fs['pct_right'], left=fs['pct_left'], color=GOLD,
                edgecolor=TEXT_COLOR, linewidth=0.8, height=0.55,
                label=f"Right ({fs['right']})")
        if fs['pct_left']  > 6:
            ax.text(fs['pct_left'] / 2, 0, f"{fs['pct_left']:.0f}%",
                    ha='center', va='center', fontsize=18, fontweight='bold', color='white')
        if fs['pct_right'] > 6:
            ax.text(fs['pct_left'] + fs['pct_right'] / 2, 0, f"{fs['pct_right']:.0f}%",
                    ha='center', va='center', fontsize=18, fontweight='bold', color=TEXT_COLOR)
        ax.set_xlim(0, 100); ax.set_ylim(-1, 1)
        ax.set_yticks([]); ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_xlabel('% of tagged actions', fontsize=13, color=TEXT_COLOR)
        ax.tick_params(colors=TEXT_COLOR, labelsize=12)
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2,
                  fontsize=13, facecolor=BG_COLOR, edgecolor=color,
                  labelcolor=TEXT_COLOR, frameon=True)
        fig.text(0.5, 0.06,
                 f"{fs['total_tagged']} tagged  ·  {fs['untagged']} untagged  ·  {n} games",
                 ha='center', va='top', fontsize=12, color=TEXT_COLOR, alpha=0.80)
    fig.savefig(OUTPUT_DIR / '09_foot_split.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 09_foot_split.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — CROSSES INTO THE BOX
# ════════════════════════════════════════════════════════════════════════════

CROSS_BOX_HIT_COLOR  = '#00E676'
CROSS_BOX_FAIL_COLOR = '#888888'

def _is_into_box(ev):
    ex, ey = end_xy(ev)
    if ex is None or ey is None: return False
    return ex >= 83.0 and 21.1 <= ey <= 78.9

def plot_crosses_into_box(acc, n):
    color = PLAYER['color']
    p = _pitch()
    fig, ax = _make_fig(
        title    = f"{PLAYER['name']}  —  Crosses into the Box",
        subtitle = (f"{COMPETITION}  ·  {PLAYER['team']}  ·  "
                    "All crosses with end-point inside the penalty area  ·  attacking left → right"),
        note     = 'Coloured arrow = successful cross  ·  Grey arrow = unsuccessful cross  ·  ● = landing zone',
    )
    all_crosses_succ = [ev for ev in acc['succ_passes'] if has_qualifier(ev, QUALIFIER_CROSS)]
    all_crosses_fail = [ev for ev in acc['fail_passes'] if has_qualifier(ev, QUALIFIER_CROSS)]
    box_succ = [ev for ev in all_crosses_succ if _is_into_box(ev)]
    box_fail = [ev for ev in all_crosses_fail if _is_into_box(ev)]

    ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                 color=color, alpha=0.06, zorder=1, linewidth=0))
    ax.add_patch(plt.Rectangle((83.0, 21.1), 17.0, 57.8,
                 fill=False, edgecolor=color, linewidth=1.2,
                 linestyle='--', alpha=0.35, zorder=2))

    if not box_succ and not box_fail:
        ax.text(50, 50, 'No crosses into box', ha='center', va='center', fontsize=20, color=TEXT_COLOR)
        _stats_box(ax, f'0 crosses into box  ·  {n} games', color=color)
    else:
        for ev in box_fail:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=CROSS_BOX_FAIL_COLOR,
                     width=1.6, headwidth=4.5, headlength=4.5, alpha=0.55, zorder=3)
            p.scatter(ex, ey, ax=ax, s=55, color='none',
                      edgecolors=CROSS_BOX_FAIL_COLOR, linewidths=1.3, alpha=0.65, zorder=4)
        for ev in box_succ:
            sx, sy = xy(ev); ex, ey = end_xy(ev)
            if None in (sx, sy, ex, ey): continue
            p.arrows(sx, sy, ex, ey, ax=ax, color=TEXT_COLOR,
                     width=3.2, headwidth=7.0, headlength=7.0, alpha=0.45, zorder=4)
            p.arrows(sx, sy, ex, ey, ax=ax, color=CROSS_BOX_HIT_COLOR,
                     width=2.0, headwidth=5.2, headlength=5.2, alpha=0.95, zorder=5)
            p.scatter(ex, ey, ax=ax, s=120, color=CROSS_BOX_HIT_COLOR,
                      edgecolors=TEXT_COLOR, linewidths=0.9, alpha=0.95, zorder=6)

        total = len(box_succ) + len(box_fail)
        succ_rate = len(box_succ) / total * 100 if total else 0.0
        legend = [Line2D([0],[0], marker='>', color=CROSS_BOX_HIT_COLOR, lw=0, markersize=16,
                         markeredgecolor=TEXT_COLOR, markeredgewidth=0.7,
                         label=f'Successful  ({len(box_succ)})'),
                  Line2D([0],[0], marker='>', color=CROSS_BOX_FAIL_COLOR, lw=0, markersize=13,
                         label=f'Unsuccessful  ({len(box_fail)})')]
        ax.legend(handles=legend, loc='lower left', fontsize=15,
                  facecolor=BG_COLOR, edgecolor=color, labelcolor=TEXT_COLOR)
        _stats_box(ax,
            f'Crosses into box   : {total}\n'
            f'Successful         : {len(box_succ)}  ({succ_rate:.0f}%)\n'
            f'Unsuccessful       : {len(box_fail)}\n'
            f'Total crosses      : {len(all_crosses_succ) + len(all_crosses_fail)}\n'
            f'Games              : {n}',
            color=color)

    fig.savefig(OUTPUT_DIR / '10_crosses_into_box.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 10_crosses_into_box.png')


# ════════════════════════════════════════════════════════════════════════════
# PLOT 11 — SUMMARY STAT BLOCK
# ════════════════════════════════════════════════════════════════════════════

def plot_summary(acc, n):
    color = PLAYER['color']

    total_pass = len(acc['succ_passes']) + len(acc['fail_passes'])
    pass_acc   = len(acc['succ_passes']) / total_pass * 100 if total_pass else 0
    total_to   = len(acc['to_succ']) + len(acc['to_fail'])
    to_sr      = len(acc['to_succ']) / total_to * 100 if total_to else 0
    _, crosses = split_passes_crosses(acc['succ_passes'])
    all_xt     = compute_pass_cross_xt(acc['succ_passes'])
    gross_xt   = sum(d['xt_end']    for d in all_xt)
    net_xt     = sum(d['xt_gained'] for d in all_xt)
    prog       = acc['prog_carries']
    spds       = [carry_speed_mps(c) for c in prog]
    avg_spd    = sum(spds) / len(spds) if spds else 0.0
    progs      = [carry_progression_m(c) for c in prog]
    avg_prog_m = sum(progs) / len(progs) if progs else 0.0
    fs         = compute_foot_split(acc)
    post_passes = [a for a in acc['post_takeon'] if a['action'] == 'pass']
    avg_xt_post = (sum(a.get('xt_gained', 0) for a in post_passes)
                   / len(post_passes) if post_passes else 0.0)
    ft_passes   = acc['pass_connections']
    ft_xt       = sum(r['xt_gained'] for r in ft_passes)
    conns       = _aggregate_connections(ft_passes, min_count=2)
    top_partner = conns[0]['recipient_name'] if conns else '–'

    stats = [
        ('Games',                    f'{n}'),
        ('──────────────────────',   '─────────'),
        ('Passes (succ/total)',       f"{len(acc['succ_passes'])} / {total_pass}  ({pass_acc:.0f}%)"),
        ('Successful crosses',        f'{len(crosses)}'),
        ('Take-ons (succ/total)',      f"{len(acc['to_succ'])} / {total_to}  ({to_sr:.0f}%)"),
        ('──────────────────────',   '─────────'),
        ('Prog. carries',             f"{len(prog)}  (into box: {len(acc['box_carries'])})"),
        ('Avg carry speed',           f'{avg_spd:.2f} m/s'),
        ('Avg carry progression',     f'{avg_prog_m:.1f} m'),
        ('──────────────────────',   '─────────'),
        ('Off-ball runs (FT+)',        f"{len(acc['runs'])}"),
        ('Chances created (open play)',f"{len(acc.get('chances', []))}  "
                                      f"(KP: {sum(1 for c in acc.get('chances',[]) if c['is_key_pass'])}  "
                                      f"GA: {sum(1 for c in acc.get('chances',[]) if c['is_goal_assist'])})"),
        ('Post-TO actions',            f"{len(acc['post_takeon'])}"),
        ('Avg xT / post-TO pass',      f'{avg_xt_post:+.4f}'),
        ('──────────────────────',   '─────────'),
        ('FT+ pass connections',       f'{len(ft_passes)}  (net xT: {ft_xt:.3f})'),
        ('Most frequent partner',      top_partner),
        ('──────────────────────',   '─────────'),
        ('Foot split (R / L tagged)', f"{fs['pct_right']:.0f}% / {fs['pct_left']:.0f}%  "
                                      f"({fs['total_tagged']} tagged)"),
        ('──────────────────────',   '─────────'),
        ('Gross xT (passes + crosses)', f'{gross_xt:.3f}'),
        ('Net xT',                    f'{net_xt:.3f}'),
    ]

    fig, ax = plt.subplots(figsize=(14, 10))
    fig.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)
    ax.axis('off')
    fig.text(0.5, 0.97, f"{PLAYER['name']}  ·  {PLAYER['team']}  —  Season Summary",
             ha='center', va='top', fontsize=26, fontweight='bold', color=color)
    fig.text(0.5, 0.91, f"{COMPETITION}  ·  {n} games",
             ha='center', va='top', fontsize=18, color=TEXT_COLOR, alpha=0.90)
    fig.text(0.5, 0.86, CREDIT_LINE,
             ha='center', va='top', fontsize=13, color=TEXT_COLOR, alpha=0.85, style='italic')

    y = 0.80
    for label, value in stats:
        is_div = label.startswith('──')
        if is_div:
            ax.plot([0.04, 0.96], [y + 0.005, y + 0.005],
                color=PITCH_LINE, lw=0.8, alpha=0.6,
                transform=ax.transAxes)
            y -= 0.018
            continue
        ax.text(0.08, y, label, transform=ax.transAxes,
                fontsize=14, color=TEXT_COLOR, fontfamily='monospace',
                fontweight='bold', va='top')
        ax.text(0.60, y, value, transform=ax.transAxes,
                fontsize=14, color=color, fontfamily='monospace', va='top')
        y -= 0.048

    fig.savefig(OUTPUT_DIR / '11_summary.png', dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close()
    print('  ✓ 11_summary.png')

print('All plot functions defined.')


# ════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════

def main():
    if not PLAYER['match_ids']:
        print('[!] No match IDs configured.'); return

    print('=' * 60)
    print(f"  {PLAYER['name']}  ({PLAYER['team']})")
    print(f"  Competition : {COMPETITION}")
    print(f"  Matches     : {len(PLAYER['match_ids'])}")
    print('=' * 60)

    # ── 1. Scrape ─────────────────────────────────────────────────────────
    print('\n[1/3] Scraping WhoScored...')
    driver = get_driver()
    acc = {
        'succ_passes': [], 'fail_passes': [],
        'to_succ': [], 'to_fail': [],
        'carries': [], 'prog_carries': [], 'box_carries': [],
        'shots': [], 'runs': [],
        'post_takeon': [],
        'pass_connections': [],
        'chances': [],
    }
    games = 0
    for mid in PLAYER['match_ids']:
        all_evs, _, name_map = scrape_match(driver, mid, PLAYER['player_id'], PLAYER['name'])
        if not all_evs:
            continue
        m = process_match(all_evs, PLAYER['player_id'], name_map)
        for k in acc:
            acc[k].extend(m.get(k, []))
        games += 1
        time.sleep(random.uniform(10, 18))
    driver.quit()

    if games == 0:
        print('\n[!] No data collected. Check match IDs.'); return

    # ── 2. Summary ────────────────────────────────────────────────────────
    print(f'\n[2/3] Scraped {games} games successfully.')

    # ── 3. Render ─────────────────────────────────────────────────────────
    print('\n[3/3] Rendering visuals...')
    plot_touch_map(acc, games)
    plot_runs_final_third(acc, games)
    plot_chance_creation(acc, games)
    plot_takeon_map(acc, games)
    plot_post_takeon_action(acc, games)
    plot_progressive_carries(acc, games)
    plot_xt_solo(acc, games)
    plot_passing_connections_final_third(acc, games)
    plot_foot_split(acc, games)
    plot_crosses_into_box(acc, games)
    plot_summary(acc, games)

    slug = PLAYER['name'].lower().replace(' ', '_')
    outputs = [
        '01_touch_map.png',
        '02_runs_final_third.png',
        '03_chance_creation.png',
        '04_takeon_map.png',
        '05_post_takeon_action.png',
        '06_progressive_carries.png',
        f'07_xt_{slug}.png',
        '08_pass_connections_ft.png',
        '09_foot_split.png',
        '10_crosses_into_box.png',
        '11_summary.png',
    ]
    print(f'\n✓ All done  —  {len(outputs)} visuals saved to: {OUTPUT_DIR.resolve()}')
    for f in outputs: print(f'    {f}')


main()

Player : Yan Diomande  (RB Leipzig)
ID     : 557000
Matches: 33
Event helpers ready.
xT helpers ready.
Filter functions ready.
Scraper ready.
Layout helpers ready.
All plot functions defined.
  Yan Diomande  (RB Leipzig)
  Competition : 2025/26 Season
  Matches     : 33

[1/3] Scraping WhoScored...
  [WS] Fetching match 1908319 ...
       ✓ Yan Diomande — 24 raw events
  [WS] Fetching match 1910607 ...
       ✓ Yan Diomande — 32 raw events
  [WS] Fetching match 1910620 ...
       ✓ Yan Diomande — 34 raw events
  [WS] Fetching match 1910639 ...
       ✓ Yan Diomande — 67 raw events
  [WS] Fetching match 1910661 ...
       ✓ Yan Diomande — 60 raw events
  [WS] Fetching match 1910626 ...
       ✓ Yan Diomande — 27 raw events
  [WS] Fetching match 1910681 ...
       ✓ Yan Diomande — 26 raw events
  [WS] Fetching match 1910689 ...
       ✓ Yan Diomande — 52 raw events
  [WS] Fetching match 1910720 ...
       ✓ Yan Diomande — 57 raw events
  [WS] Fetching match 1910738 ...
       ✓ Yan Dioma